
# 🎨 ClawSouls — Avatar API Server (FastAPI + Cloudflared)

Sobe um servidor **FastAPI** no Colab que gera avatares por requisição HTTP.

**Fluxo:**
1. Rode este notebook (com GPU)
2. Copie a URL pública do Cloudflared
3. Envie POST `/generate` com os atributos da soul
4. Receba a imagem gerada em base64

---



## Pré-requisitos

- `Runtime > Change runtime type > T4 GPU`
- Não precisa do repositório clonado (tudo é self-contained)


In [ ]:

# ═══════════════════════════════════════════════════════════
# TOGGLE DE MODELO
# ═══════════════════════════════════════════════════════════

MODELO = "sdxl"          # ← "sdxl" ou "turbo"
SECRET_TOKEN = "cs-secret-2026"  # ← Troque para algo seguro!
TOTAL_STEPS = 15          # Override global de steps
TOTAL_GUIDANCE = 7.5      # Override global de guidance

CATALOGO = {
    "turbo": {
        "model_id": "Tongyi-MAI/Z-Image-Turbo",
        "width": 512, "height": 768, "variant": "fp16",
        "default_steps": 4, "default_guidance": 1.0,
    },
    "sdxl": {
        "model_id": "stabilityai/stable-diffusion-xl-base-1.0",
        "width": 512, "height": 768, "variant": "fp16",
        "default_steps": 25, "default_guidance": 7.5,
    },
}

assert MODELO in CATALOGO, f"Use: {list(CATALOGO.keys())}"
cfg = CATALOGO[MODELO]
MODEL_ID = cfg["model_id"]

print(f"🔧 Modelo: {MODELO} ({MODEL_ID})")
print(f"   Steps: {TOTAL_STEPS}, Guidance: {TOTAL_GUIDANCE}, Token: {SECRET_TOKEN[:4]}...")


In [ ]:

# Escreve o servidor num arquivo .py para que o uvicorn consira importá-lo corretamente.
# O notebook não serve como módulo para o uvicorn (--reload).

import textwrap

server_code = textwrap.dedent('''
    from urllib.parse import parse_qs
    import io, base64, time, os, re
    from datetime import datetime
    from typing import Optional

    from pydantic import BaseModel
    from fastapi import FastAPI, HTTPException, Header, Request
    import torch
    from diffusers import AutoPipelineForText2Image

    # ── Config (injetadas pelo notebook via env vars) ──────────
    MODELO = os.environ.get('CLAWSOULS_MODELO', 'sdxl')
    SECRET_TOKEN = os.environ.get('CLAWSOULS_SECRET', 'cs-secret-2026')
    TOTAL_STEPS = int(os.environ.get('CLAWSOULS_STEPS', '15'))
    TOTAL_GUIDANCE = float(os.environ.get('CLAWSOULS_GUIDANCE', '7.5'))
    MODEL_ID = os.environ.get('CLAWSOULS_MODEL_ID', 'stabilityai/stable-diffusion-xl-base-1.0')
    MODEL_WIDTH = int(os.environ.get('CLAWSOULS_WIDTH', '512'))
    MODEL_HEIGHT = int(os.environ.get('CLAWSOULS_HEIGHT', '768'))

    # ── Prompt Engine ────────────────────────────────────────
    EMOJI_HINTS = {
        '🔬': 'scientific goggles, lab coat details',
        '🕵️': 'detective hat, trench coat',
        '🌟': 'sparkles, star-shaped accessories',
        '⚡': 'electric energy aura, lightning motifs',
        '🧘': 'lotus position, meditation beads, serene',
        '🤖': 'mechanical parts, circuit patterns',
        '🏴‍☠️': 'pirate bandana, adventurous look',
        '💻': 'techwear, holographic screen elements',
        '🎤': 'microphone, stage lights, glamorous',
        '🌳': 'nature elements, leaves, organic flowing design',
        '🕶️': 'sunglasses, cool demeanor',
        '😈': 'mischievous grin, horns, dark aesthetic',
        '👽': 'alien features, cosmic glow',
        '🐉': 'dragon scales, mythical aura',
        '🦊': 'fox ears, cunning expression',
        '🐱': 'cat ears, playful whiskers',
        '👁️': 'mystical third eye, all-seeing aura',
        '💀': 'skull motifs, dark mysticism',
        '🎭': 'theater mask, dramatic duality',
    }

    DOMAIN_ACCENTS = {
        'tech': 'circuit patterns, holographic UI elements',
        'philosophy': 'ancient scrolls, ethereal glow',
        'science': 'molecular structures, lab equipment details',
        'arts': 'paint splashes, creative chaos',
        'history': 'ancient runes, time-worn textures',
        'literature': 'floating text, book pages',
        'pop-culture': 'retro gaming elements, neon signs',
        'sports': 'athletic build, competitive energy',
        'business': 'sharp suit, corporate confidence',
        'psychology': 'thoughtful gaze, abstract mind visuals',
    }

    def build_prompt(soul: dict) -> str:
        creature = soul.get('creature', 'mysterious entity')
        vibe = soul.get('vibe', 'enigmatic')
        emoji = soul.get('emoji', '')
        humor = soul.get('humor', 50)
        formality = soul.get('formality', 50)
        vibe_style = soul.get('vibeStyle', 'concise')
        knowledge_domains = soul.get('knowledgeDomains', [])
        emotional_range = soul.get('emotionalRange', 50)
        agreeableness = soul.get('agreeableness', 50)
        extraversion = soul.get('extraversion', 50)
        openness = soul.get('openness', 70)
        neuroticism = soul.get('neuroticism', 30)
        is_high_formality = formality > 65
        is_playful = humor > 65
        is_minimal = vibe_style == 'minimal'
        is_concise = vibe_style == 'concise'
        is_dramatic = emotional_range > 75 or vibe_style == 'dramatic'
        is_tech = any(d in ('tech', 'science') for d in knowledge_domains)
        art_style = 'cyberpunk digital illustration'
        if is_high_formality:
            art_style = 'elegant digital painting, Renaissance lighting'
        elif is_playful:
            art_style = 'colorful anime-inspired digital art, vibrant'
        elif is_minimal:
            art_style = 'minimalist vector art, clean lines, geometric'
        elif is_tech:
            art_style = 'sci-fi concept art, holographic elements'
        elif is_dramatic:
            art_style = 'cinematic digital painting, dramatic chiaroscuro lighting'
        atmosphere = 'dark atmospheric background with neon accents'
        if agreeableness > 70:
            atmosphere = 'warm, inviting background with soft golden light'
        elif neuroticism > 60:
            atmosphere = 'unstable, glitching background with fractured light'
        elif extraversion > 70:
            atmosphere = 'dynamic, energetic background with bold colors'
        elif openness > 75:
            atmosphere = 'dreamy, surreal background with cosmic elements'
        expression = 'calm, confident expression'
        if neuroticism > 60:
            expression = 'tense, alert expression'
        elif extraversion > 70:
            expression = 'bright, engaging smile'
        elif agreeableness > 70:
            expression = 'gentle, warm expression'
        elif openness > 70:
            expression = 'curious, contemplative gaze'
        elif humor > 70:
            expression = 'sly, playful smirk'
        descriptors = [creature, vibe]
        if expression != 'calm, confident expression':
            descriptors.append(expression)
        descriptors.append(art_style)
        if not is_concise and not is_minimal:
            descriptors.append('vibe: ' + vibe_style)
        if emoji and emoji in EMOJI_HINTS:
            descriptors.append(EMOJI_HINTS[emoji])
        for domain in knowledge_domains:
            if domain in DOMAIN_ACCENTS:
                descriptors.append(DOMAIN_ACCENTS[domain])
        descriptors.append('unique, one-of-a-kind character design')
        prompt = (
            'close-up portrait, centered, detailed face, '
            + 'professional character art of ' + creature + ', '
            + ', '.join(descriptors[1:]) + ', '
            + atmosphere + ', highly detailed, 4k, masterpiece'
        )
        return prompt.strip()

    def build_negative_prompt() -> str:
        return (
            'blurry, low quality, deformed, ugly, duplicate, disfigured, '
            'bad anatomy, bad proportions, extra limbs, mutated hands, '
            'text, watermark, signature, logo, '
            'photorealistic, 3d render, '
            'nude, NSFW, gore'
        )

    # ── FastAPI App ────────────────────────────────────────
    class GenerateRequest(BaseModel):
        name: str
        creature: Optional[str] = 'mysterious entity'
        vibe: Optional[str] = 'enigmatic'
        emoji: Optional[str] = ''
        humor: Optional[int] = 50
        formality: Optional[int] = 50
        emojiUsage: Optional[int] = 20
        knowledgeDomains: Optional[list] = []
        emotionalRange: Optional[int] = 50
        vibeStyle: Optional[str] = 'concise'
        agreeableness: Optional[int] = 50
        extraversion: Optional[int] = 50
        openness: Optional[int] = 70
        neuroticism: Optional[int] = 30
        steps: Optional[int] = None
        guidance: Optional[float] = None
        custom_prompt: Optional[str] = None

    app = FastAPI(title='ClawSouls Avatar API', version='2.0')

    def _check_token(request: Request) -> bool:
        auth = request.headers.get('Authorization', '')
        if auth == 'Bearer ' + SECRET_TOKEN:
            return True
        qs = parse_qs(request.url.query)
        if qs.get('token', [''])[0] == SECRET_TOKEN:
            return True
        if request.headers.get('X-Token') == SECRET_TOKEN:
            return True
        return False

    @app.get('/health')
    def health():
        return {'status': 'ok', 'model': MODELO, 'model_id': MODEL_ID}

    @app.get('/models')
    def list_models():
        return {'available': {}, 'current': MODELO}

    @app.post('/generate')
    async def generate(request: Request, req: GenerateRequest):
        if not _check_token(request):
            raise HTTPException(status_code=401, detail='Unauthorized')
        if req.custom_prompt:
            prompt = req.custom_prompt
        else:
            d = req.dict()
            d.pop('steps', None)
            d.pop('guidance', None)
            d.pop('custom_prompt', None)
            prompt = build_prompt(d)
        negative = build_negative_prompt()
        steps = req.steps if req.steps else TOTAL_STEPS
        guidance = req.guidance if req.guidance else TOTAL_GUIDANCE
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Generating: {req.name} ({steps} steps)")
        start = time.time()
        generator = torch.Generator(device='cuda').manual_seed(int(time.time() * 1000) % (2**32))
        image = pipe(
            prompt=prompt, negative_prompt=negative,
            num_inference_steps=steps, guidance_scale=guidance,
            width=MODEL_WIDTH, height=MODEL_HEIGHT, generator=generator,
        ).images[0]
        elapsed = time.time() - start
        buf = io.BytesIO()
        image.save(buf, format='PNG')
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode('utf-8')
        safe_name = ''.join(c if c.isalnum() or c in '._-' else '_' for c in req.name.lower().strip())
        print(f"   ✅ {req.name} → {elapsed:.1f}s")
        return {
            'name': req.name, 'slug': safe_name, 'prompt': prompt,
            'negative_prompt': negative, 'seed': int(generator.initial_seed()),
            'steps': steps, 'guidance': guidance, 'model': MODELO,
            'elapsed_s': round(elapsed, 1), 'image_base64': img_b64,
        }

    # ── Lifespan (model loading) ─────────────────────────────
    from contextlib import asynccontextmanager

    @asynccontextmanager
    async def lifespan(app):
        global pipe
        variant = os.environ.get('CLAWSOULS_VARIANT', 'fp16')
        pipe = AutoPipelineForText2Image.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16 if variant == 'fp16' else torch.float32,
            variant=variant if variant != 'fp32' else None,
            use_safetensors=True,
        )
        pipe.enable_attention_slicing()
        if hasattr(pipe, 'enable_vae_tiling'):
            pipe.enable_vae_tiling()
        pipe.to('cuda')
        # Fix: forçar VAE no mesmo dtype pra evitar RuntimeError de mixed precision
        if hasattr(pipe, 'vae'):
            pipe.vae.to(torch_device='cuda', torch_dtype=torch.float16)
        print('✅ Modelo carregado: ' + MODEL_ID)
        yield
        del pipe
        torch.cuda.empty_cache()
        print('🛑 Servidor encerrado')

    app.router.lifespan_context = lifespan

''')

# Injecta config do notebook como env vars pro server.py ler
import os as _os
_os.environ['CLAWSOULS_MODELO'] = MODELO
_os.environ['CLAWSOULS_SECRET'] = SECRET_TOKEN
_os.environ['CLAWSOULS_STEPS'] = str(TOTAL_STEPS)
_os.environ['CLAWSOULS_GUIDANCE'] = str(TOTAL_GUIDANCE)
_os.environ['CLAWSOULS_MODEL_ID'] = MODEL_ID
_os.environ['CLAWSOULS_WIDTH'] = str(cfg['width'])
_os.environ['CLAWSOULS_HEIGHT'] = str(cfg['height'])
_os.environ['CLAWSOULS_VARIANT'] = cfg['variant']

# Grava o server.py
with open('/content/server.py', 'w') as f:
    f.write(server_code)

print('✅ server.py gravado!')


In [ ]:

# Inicia uvicorn rodando server.py em background
uvicorn_proc = !nohup python3 -m uvicorn server:app --host 0.0.0.0 --port 8000 --log-level warning > /tmp/uvicorn.log 2>&1 &

# Healthcheck: espera o servidor responder antes de subir o tunnel
import time, urllib.request

print('⏳ Esperando servidor iniciar...')
server_ok = False
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        server_ok = True
        print('✅ Servidor ativo!')
        break
    except Exception as e:
        time.sleep(1)

if not server_ok:
    print('⚠️  Servidor não respondeu em 30s!')
    !cat /tmp/uvicorn.log


---

## Túnel Cloudflared


In [ ]:

import subprocess

# Inicia cloudflared tunnel em background
cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Espera a URL do tunnel aparecer nos logs
tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print('=' * 60)
    print('🌐 TÚNEL ATIVO!')
    print(f'📎 URL pública: {tunnel_url}')
    print()
    print('Endpoints:')
    print(f'  GET  {tunnel_url}/health')
    print(f'  GET  {tunnel_url}/models')
    print(f'  POST {tunnel_url}/generate')
    print('=' * 60)
    print()
    print('⚠️  TOKEN: ' + SECRET_TOKEN)
    print()
    print('📋 Copie esta URL e cole aqui no chat para eu usar!')
else:
    print('⚠️  Não foi possível obter a URL do tunnel.')
    print('   Verifique: cloudflared_proc.wait()')


---

## Como usar

Cole a URL do tunnel aqui no chat. Eu faço a requisição e devolvo a imagem!

```
POST /generate
{
  "name": "Kira",
  "creature": "AI / Idol",
  "emoji": "🎤",
  "vibe": "Ídolo pop digital.",
  "humor": 60,
  "vibeStyle": "expressive"
}
```
